# ST456 Colab 一键运行 Notebook（VS Code 本地连接版）

适用于 VS Code 通过 SSH 连接 Colab 运行时的场景：

- 项目文件已在 Colab VM 上（通过 VS Code 上传）
- 自动检测项目目录，也可手动指定
- 结果直接在 VS Code 文件浏览器中查看
- E1-E5 是主实验，retrieval 只作为 appendix

In [ ]:
# ===== 参数区：运行前先改这里 =====

# 项目目录：留空则自动检测，或手动指定完整路径
PROJECT_ROOT = ''  # 例如 '/content/ST456group/text'

QUICK_VALIDATION = True
RUN_FULL_MAINLINE = True
RUN_APPENDIX_RETRIEVAL = False

print('参数已加载。')

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

def run(cmd, cwd=None):
    import html
    from collections import deque
    from IPython.display import HTML, display

    print(f'\n>>> {cmd}', flush=True)
    status_handle = display(
        HTML(f"<pre>>>> {html.escape(cmd)}\n{html.escape('启动中...')}</pre>"),
        display_id=True,
    )
    tail = deque(maxlen=80)

    def update_status(text):
        safe_cmd = html.escape(cmd)
        safe_text = html.escape(text or '运行中...')
        status_handle.update(HTML(f"<pre>>>> {safe_cmd}\n{safe_text}</pre>"))

    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    current = ''
    while True:
        char = process.stdout.read(1)
        if char == '' and process.poll() is not None:
            break
        if not char:
            continue
        if char == '\r':
            line = current.strip()
            if line:
                update_status(line[-400:])
            current = ''
            continue
        if char == '\n':
            line = current.strip()
            if line:
                tail.append(line)
                update_status(line[-400:])
            current = ''
            continue
        current += char

    final_line = current.strip()
    if final_line:
        tail.append(final_line)
        update_status(final_line[-400:])

    process.wait()
    if process.returncode != 0:
        recent_output = '\n'.join(tail) or '没有捕获到可用输出。'
        update_status(f'失败（exit code {process.returncode}）')
        raise RuntimeError(
            f'命令执行失败（exit code {process.returncode}）：{cmd}\n'
            f'最近输出：\n{recent_output}'
        )

    update_status('完成')

def clear_gpu_memory(stage=''):
    import gc

    gc.collect()
    try:
        import torch
    except ImportError:
        print(f'[GPU 清理] {stage or "当前阶段"}: torch 未安装，已跳过。')
        return

    if not torch.cuda.is_available():
        print(f'[GPU 清理] {stage or "当前阶段"}: 当前没有可用 CUDA，已跳过。')
        return

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, 'ipc_collect'):
        torch.cuda.ipc_collect()
    reserved_mb = torch.cuda.memory_reserved() / (1024 ** 2)
    allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    print(
        f'[GPU 清理] {stage or "当前阶段"}: '
        f'allocated={allocated_mb:.1f} MB, reserved={reserved_mb:.1f} MB'
    )

def find_project_root():
    """自动检测 text 目录。"""
    search_roots = [Path.cwd(), Path('/content'), Path.home()]
    for root in search_roots:
        for candidate in root.rglob('text'):
            if candidate.is_dir() and (candidate / 'src').exists():
                return candidate
    return None

if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    print('自动检测项目目录...')
    project_root = find_project_root()
    if project_root is None:
        raise FileNotFoundError(
            '未找到 text 目录。\n'
            '请在参数区手动设置 PROJECT_ROOT。'
        )

if not project_root.exists():
    raise FileNotFoundError(f'项目目录不存在: {project_root}')

os.chdir(project_root)
src_root = project_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
print('项目目录:', Path.cwd())
print('已加入 Python 路径:', src_root)
run(f'{sys.executable} -m pip install -r requirements.txt')
print('环境准备完成。')

## 数据准备与 token 预算检查

In [ ]:
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]
for experiment_id, config_path in token_stat_configs:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')

print('Token 统计文件:')
for path in sorted(Path('artifacts/eval').glob('token_stats_*.json')):
    print('-', path)

## 主实验训练

E1 一定会跑。如果 `RUN_FULL_MAINLINE = True`，会继续跑 E2-E5。

In [ ]:
from novel_continuation.training import load_training_config

main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
    ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
    ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ('E5W', 'configs/e5_distilgpt2_structured_aux_ranking_wide.yaml', 'artifacts/e5_aux_ranking_wide'),
]

for experiment_id, config_path, _output_dir in main_experiments:
    print(f'----- {experiment_id}: 开始训练 -----')
    run(f'{sys.executable} scripts/train_experiment.py --config {config_path} --seed 42')
    clear_gpu_memory(f'{experiment_id} 训练结束')

print('主实验训练完成。')
for experiment_id, config_path, _output_dir in main_experiments:
    resolved = load_training_config(Path(config_path))
    print(f'- {experiment_id}: output_dir={resolved["output_dir"]}')


## 3-seed 样本生成与自动评估（人评可选）

这一步会为每个已经训练完成的主实验生成：

- `generated_samples_*_seed*.jsonl`
- `metrics_*_seed*.csv`
- `metrics_*_summary.csv`
- `human_eval_*.csv`（可选）

In [ ]:
import json
eval_output_dir = Path('artifacts/eval')
eval_output_dir.mkdir(parents=True, exist_ok=True)

for experiment_id, _config_path, model_dir in main_experiments:
    print(f'\n----- {experiment_id}: 3-seed 自动评估 -----')
    run(
        f'{sys.executable} scripts/run_eval_3seed.py ' \
        f'--experiment-id {experiment_id.lower()} ' \
        f'--model-dir {model_dir} ' \
        f'--output-dir {eval_output_dir}'
    )
    clear_gpu_memory(f'{experiment_id} 评测结束')

print('\n3-seed 自动评估文件已生成。')
for path in sorted(eval_output_dir.glob('*')):
    print('-', path)

print('\n比较 E5 与 E5W 的 validation_main_loss：')
for experiment_id, _config_path, model_dir in main_experiments:
    if experiment_id not in {'E5', 'E5W'}:
        continue
    training_config = Path(model_dir) / 'training_config.json'
    if not training_config.exists():
        print(f'- {experiment_id}: 未找到 {training_config}')
        continue
    payload = json.loads(training_config.read_text(encoding='utf-8'))
    validation = payload.get('metadata', {}).get('validation', {})
    print(f'- {experiment_id}: validation_main_loss={validation.get("validation_main_loss")}, validation_perplexity={validation.get("validation_perplexity")}')


## Appendix：retrieval 附加实验

只有在 `RUN_APPENDIX_RETRIEVAL = True` 时才会执行这一块。

In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
    run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
    run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
    # human-eval 导出保留为可选附加步骤，本轮默认不执行
    print('retrieval 附加实验已完成。')
else:
    print('已跳过 retrieval 附加实验。')

## 清理 checkpoint 并汇总结果

清理中间 checkpoint 节省磁盘空间，结果文件在 `artifacts/eval/` 目录下，可直接通过 VS Code 文件浏览器查看。

In [ ]:
for ckpt_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
    if ckpt_dir.is_dir():
        shutil.rmtree(ckpt_dir)
        print(f'已清理: {ckpt_dir}')

print('\n===== 全部完成 =====')
print(f'结果目录: {Path.cwd() / "artifacts" / "eval"}')
print('\n生成的文件:')
for path in sorted(Path('artifacts/eval').glob('*')):
    size_kb = path.stat().st_size / 1024
    print(f'  {path.name}  ({size_kb:.1f} KB)')
print('\n可直接在 VS Code 文件浏览器中查看 artifacts/eval/ 目录。')